# Modeling and evaluation: predicting the price of a user's next reviewed game

*An earlier version of our modeling and evaluation. The current results are on the [Modeling](modeling.ipynb) and [Evaluation](evaluation.ipynb) pages.*

**Task.** For each review a user writes, predict the list price of that game using only what was known before
that day: the user's earlier reviews, what other users reviewed earlier, and the calendar. The target game's
identity and attributes are never used.

**Protocol.** Reviews are split by date into train (oldest 80%), validation (next 10%) and test (newest 10%).
Models are fit on train and compared on validation; the chosen models are refit on train + validation and
scored on test once. Metrics are MAE, MSE and MSLE (definitions in the intro).

In [1]:
import ast
import datetime as dt
import re
from functools import lru_cache
from pathlib import Path

import hvplot.polars
import numpy as np
import polars as pl
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from nltk.stem import PorterStemmer
from scipy.optimize import minimize, nnls
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from utils import LABELS, load_games

pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_fmt_str_lengths(60)
DATA = Path("data")
SEED = 0

---

## 1. Data

Starting from the review crawl, we:

1. drop exact duplicate rows (the crawl collected about 831k reviews twice, at different listing pages);
2. keep reviews with a numeric `user_id` (59% of reviews have none; `username` is not a safe substitute,
   since about 16% of rows share a username with a different id);
3. keep reviews of games with a usable list price;
4. keep a user's first review of each game.

In [2]:
TOP_G = ["Indie", "Action", "Adventure", "Casual", "Strategy", "Simulation", "RPG", "Free to Play",
         "Early Access", "Sports", "Massively Multiplayer", "Racing"]
GEN = [f"g_{g.lower().replace(' ', '_')}" for g in TOP_G]

games = load_games(DATA / "steam_games.json")
meta = games.drop_nulls("price").select(
    "product_id", "price", "release",
    *[pl.col("genres").list.contains(g).cast(pl.Float64).alias(c) for g, c in zip(TOP_G, GEN)])

raw = pl.read_parquet(DATA / "cache" / "reviews.parquet")
dedup = raw.unique()
with_id = dedup.filter(pl.col("user_id").is_not_null())
labelled = with_id.join(meta, on="product_id")
ev = (labelled.sort("user_id", "date", "product_id")
              .unique(["user_id", "product_id"], keep="first", maintain_order=True)
              .with_columns(day=pl.col("date").dt.date(),
                            lp=pl.col("price").log1p(),
                            free=(pl.col("price") == 0).cast(pl.Float64),
                            lh=pl.col("hours").fill_null(0).log1p(),
                            age=(pl.col("date").dt.date() - pl.col("release")).dt.total_days().clip(-365, 5000) / 365))

pl.DataFrame({"step": ["raw reviews", "exact duplicates removed", "with user_id", "game has a price",
                       "first review per (user, game)"],
              "rows": [raw.height, dedup.height, with_id.height, labelled.height, ev.height]})

step,rows
str,i64
"""raw reviews""",7793069
"""exact duplicates removed""",6961672
"""with user_id""",2833192
"""game has a price""",2806118
"""first review per (user, game)""",2805451


### Temporal split and prediction targets

The split points are the 80th and 90th percentiles of review dates. A review is a **prediction target** when
its user reviewed something on an earlier day, so there is a history to predict from. Reviews written on the
same day as the target are not part of its history: the crawl gives dates, not times, so their order within a
day is unknown.

In [3]:
T1, T2 = ev["date"].quantile(0.8), ev["date"].quantile(0.9)
SPLITS = ["train", "val", "test"]
ev = ev.with_columns(split=pl.when(pl.col("date") < T1).then(pl.lit("train"))
                             .when(pl.col("date") < T2).then(pl.lit("val")).otherwise(pl.lit("test")))
print(f"train < {T1.date()} <= val < {T2.date()} <= test")

weekly = (ev.sort("date").group_by_dynamic("date", every="1w", group_by="split").agg(reviews=pl.len())
            .filter(pl.col("date") >= dt.datetime(2014, 1, 1)))
weekly.hvplot.area("date", "reviews", by="split", stacked=False, alpha=0.6, width=760, height=240,
                   title="Reviews per week, coloured by split")

train < 2017-07-07 <= val < 2017-11-06 <= test


:NdOverlay   [split]
   :Area   [date]   (reviews)

---

## 2. Features: the user's state before the day

Every feature describes what was observable **before the target's day**. One pattern builds them all:

1. sort a user's reviews by date and compute running statistics that include each review;
2. keep the last running value of each day (the user's state at the end of that day);
3. shift by one day, so each review sees the state at the end of the user's previous review day.

Shifting whole days, not single rows, is what keeps same-day reviews out of each other's history.

| group | features |
|---|---|
| history | number of past reviews, mean price and mean log price, free share, spread, max, median of the last 10, exponentially weighted means, previous day's price |
| taste | mean hours played, mean popularity and age (years since release) of past games, share of past games in each of 12 genres |
| timing | days since the previous review, days since the first review, month, weekday |
| item-to-item (Markov) | what other users reviewed next after the user's last game, on earlier days (below) |
| global | mean price, mean log price and free share of all reviews in the previous 7 and 30 days |

In [4]:
u = pl.col("user_id")
ev = ev.with_columns(ipop=pl.int_range(pl.len()).over("product_id", order_by="date").log1p())

SUMS = ["lp", "price", "free", "lh", "ipop", *GEN]
running = ev.with_columns(
    n=pl.int_range(1, pl.len() + 1).over(u),
    n_age=pl.col("age").is_not_null().cum_sum().over(u),
    s_age=pl.col("age").fill_null(0).cum_sum().over(u),
    s_lp2=(pl.col("lp") ** 2).cum_sum().over(u),
    mx=pl.col("price").cum_max().over(u),
    med10=pl.col("price").rolling_median(10, min_samples=1).over(u),
    **{f"ewm{a}": pl.col("price").ewm_mean(alpha=a).over(u) for a in (0.1, 0.3, 0.5, 0.7, 0.9)},
    ewm_lp=pl.col("lp").ewm_mean(alpha=0.1).over(u),
    first_day=pl.col("day").first().over(u),
    **{f"s_{c}": pl.col(c).cum_sum().over(u) for c in SUMS},
)
STATE = ["n", "n_age", "s_age", "s_lp2", "mx", "med10", "ewm_lp", "first_day",
         *[c for c in running.columns if c.startswith("ewm0")], *[f"s_{c}" for c in SUMS]]
before = (running.group_by("user_id", "day", maintain_order=True)
                 .agg(pl.col(STATE).last(), last_p=pl.col("price").mean(), last_lp=pl.col("lp").mean(),
                      last_item=pl.col("product_id").last())
                 .with_columns(pl.col(STATE + ["last_p", "last_lp", "last_item"]).shift(1).over(u),
                               prev_day=pl.col("day").shift(1).over(u)))

n = pl.col("n")
X = (ev.select("user_id", "product_id", "day", "split", "price", "lp", "products")
       .join(before, on=["user_id", "day"])
       .filter(n.is_not_null())
       .with_columns(logn=n.log(), m_p=pl.col("s_price") / n, m_lp=pl.col("s_lp") / n,
                     free_sh=pl.col("s_free") / n,
                     sd_lp=(pl.col("s_lp2") / n - (pl.col("s_lp") / n) ** 2).clip(0, None).sqrt(),
                     m_lh=pl.col("s_lh") / n, m_ipop=pl.col("s_ipop") / n, m_age=pl.col("s_age") / pl.col("n_age"),
                     gap=(pl.col("day") - pl.col("prev_day")).dt.total_days().log1p(),
                     tenure=(pl.col("day") - pl.col("first_day")).dt.total_days().log1p(),
                     month=pl.col("day").dt.month(), dow=pl.col("day").dt.weekday(),
                     lprod=pl.col("products").log1p(),
                     **{f"sh_{c}": pl.col(f"s_{c}") / n for c in GEN}))
X.group_by("split").agg(targets=pl.len(), users=pl.col("user_id").n_unique(), mean_price=pl.col("price").mean(),
                        median_price=pl.col("price").median(), free=(pl.col("price") == 0).mean(),
                        median_history=pl.col("n").median()).sort(pl.col("split").replace_strict(SPLITS, [0, 1, 2]))

split,targets,users,mean_price,median_price,free,median_history
str,u32,u32,f64,f64,f64,f64
"""train""",959337,332006,16.114,14.990,0.164,2.000
"""val""",150066,93055,18.856,14.990,0.174,3.000
"""test""",143205,101024,19.459,14.990,0.142,2.000


**Leakage check.** Recompute two features straight from the raw reviews for 2,000 random targets, keeping
only that user's reviews from strictly earlier days, and compare.

In [5]:
check = X.sample(2000, seed=SEED).select("user_id", "day", "n", "m_lp").unique(["user_id", "day"])
direct = (check.join(ev.select("user_id", pl.col("day").alias("d"), "lp"), on="user_id")
               .filter(pl.col("d") < pl.col("day"))
               .group_by("user_id", "day").agg(n_direct=pl.len(), m_lp_direct=pl.col("lp").mean()))
c = check.join(direct, on=["user_id", "day"])
assert c.height == check.height
assert (c["n"] == c["n_direct"]).all() and np.allclose(c["m_lp"], c["m_lp_direct"])
print("history features match a direct strictly-earlier recomputation on", c.height, "targets")

history features match a direct strictly-earlier recomputation on 1999 targets


### Global context and item-to-item transitions

**Global context** is the mean price, mean log price and free share of everything reviewed in the 7 and 30
days before the target day. It tracks drift: the mean reviewed price rises from about \$16 in train to about
\$19 in test.

**Item-to-item transitions** are a first-order Markov chain over games, the sequential model from lecture.
For the user's last game *i*, they summarise the prices of what other users reviewed next after *i*. Only
transitions on earlier days count, and the averages are shrunk toward the training mean with weight $K=20$:
$\hat\mu_i = (\sum y + K\bar y)/(n_i + K)$, the same shrinkage as the hw2 rating model.

Our first version used every training transition, leaving out only the row's own pair. It looked strong on
train and fell apart on validation, because transitions from the same weeks leak the period's popular
releases. Counting only earlier days fixes it, and train, validation and test rows get the same treatment.

In [6]:
gd = (ev.group_by("day").agg(cnt=pl.len(), sp=pl.col("price").sum(), slp=pl.col("lp").sum(), sf=pl.col("free").sum())
        .sort("day").upsample("day", every="1d").fill_null(0))
for w in (7, 30):
    roll = lambda c: pl.col(c).rolling_sum(w, min_samples=1)
    gd = gd.with_columns(**{f"glob_p{w}": (roll("sp") / roll("cnt")).shift(1),
                            f"glob_lp{w}": (roll("slp") / roll("cnt")).shift(1),
                            f"glob_free{w}": (roll("sf") / roll("cnt")).shift(1)})
X = X.join(gd.drop("cnt", "sp", "slp", "sf"), on="day", how="left")

K = 20
train_rows = X.filter(pl.col("split") == "train")
PRIOR = {"lp": train_rows["lp"].mean(), "p": train_rows["price"].mean(), "f": (train_rows["price"] == 0).mean()}
trans = (X.group_by("last_item", "day")
          .agg(t_n=pl.len(), t_lp=pl.col("lp").sum(), t_p=pl.col("price").sum(), t_f=(pl.col("price") == 0).sum())
          .sort("last_item", "day")
          .with_columns(pl.col("t_n", "t_lp", "t_p", "t_f").cum_sum().shift(1).over("last_item").fill_null(0)))
X = (X.join(trans, on=["last_item", "day"], how="left")
      .with_columns(tr_lp=(pl.col("t_lp") + K * PRIOR["lp"]) / (pl.col("t_n") + K),
                    tr_p=(pl.col("t_p") + K * PRIOR["p"]) / (pl.col("t_n") + K),
                    tr_f=(pl.col("t_f") + K * PRIOR["f"]) / (pl.col("t_n") + K),
                    tr_logn=pl.col("t_n").log1p()))

HIST = ["logn", "m_p", "m_lp", "free_sh", "sd_lp", "mx", "med10", "ewm0.1", "ewm_lp", "last_p", "last_lp"]
TASTE = ["m_lh", "m_ipop", "m_age", *[f"sh_{c}" for c in GEN]]
TIMING = ["gap", "tenure", "month", "dow"]
MARKOV = ["tr_lp", "tr_p", "tr_f", "tr_logn"]
GLOBAL = ["glob_p7", "glob_lp7", "glob_free7", "glob_p30", "glob_lp30", "glob_free30"]
FEATURES = HIST + TASTE + TIMING + MARKOV + GLOBAL
print(len(FEATURES), "features")

40 features


---

## 3. Evaluation code

Each metric has a different best constant: the **median** minimises MAE, the **mean** minimises MSE, and
$\exp(\overline{\log(1+y)})-1$ (a geometric-style mean) minimises MSLE. So no single prediction is best on all
three. The baselines produce one prediction each. For the main models we train one model per metric, each
with the loss that matches it.

In [7]:
def scores(y, p):
    p = np.clip(p, 0, None)
    return {"MAE": np.mean(np.abs(y - p)), "MSE": np.mean((y - p) ** 2),
            "MSLE": np.mean((np.log1p(y) - np.log1p(p)) ** 2)}

def table(preds, y):
    return pl.DataFrame([{"model": k, **scores(y, p)} for k, p in preds.items()])

tr, va, te = (X.filter(pl.col("split") == s) for s in SPLITS)
y_tr, y_va, y_te = (d["price"].to_numpy() for d in (tr, va, te))

---

## 4. Baselines

From the proposal: the last price, the user's median, a recency-weighted mean (EWA), a linear regression on
user features, and a Markov chain. We add constants and a shrunk user mean (hw2's shrinkage model).

The EWA weight $\alpha$ and the shrinkage strength are tuned on validation.

In [8]:
ewa = table({f"alpha={a}": va[f"ewm{a}"].to_numpy() for a in (0.1, 0.3, 0.5, 0.7, 0.9)}, y_va)

g = np.log1p(y_tr).mean()
def shrunk(d, k):
    return np.expm1(g + d["n"].to_numpy() / (d["n"].to_numpy() + k) * (d["m_lp"].to_numpy() - g))
shrink = table({f"k={k}": shrunk(va, k) for k in (0, 1, 2, 5, 10, 20, 50)}, y_va)
pl.concat([ewa.with_columns(pl.lit("EWA").alias("family")), shrink.with_columns(pl.lit("shrunk").alias("family"))])

model,MAE,MSE,MSLE,family
str,f64,f64,f64,str
"""alpha=0.1""",12.948,375.114,1.955,"""EWA"""
"""alpha=0.3""",13.091,379.543,1.971,"""EWA"""
"""alpha=0.5""",13.357,391.849,2.004,"""EWA"""
"""alpha=0.7""",13.758,413.662,2.070,"""EWA"""
"""alpha=0.9""",14.316,449.038,2.225,"""EWA"""
"""k=0""",13.228,396.432,1.957,"""shrunk"""
"""k=1""",12.635,348.518,1.577,"""shrunk"""
"""k=2""",12.795,355.974,1.521,"""shrunk"""
"""k=5""",13.124,368.758,1.520,"""shrunk"""


Small $\alpha$, which weights a long history almost evenly, is best on every metric. Heavier recency only adds
noise. For the shrunk mean, the best $k$ is large: one user's history is weak evidence compared with the
population. We keep EWA with $\alpha=0.1$ and shrinkage with the $k$ that minimises validation MSLE.

In [9]:
K_SHRINK = int(shrink.sort("MSLE")["model"][0].split("=")[1])
A_EWA = 0.1

bucket_of = lambda c: pl.col(c).cut([0, 4.99, 9.99, 19.99, 29.99, 59.99], labels=LABELS).cast(pl.Enum(LABELS))
chain = (tr.with_columns(state=bucket_of("last_p"))
           .group_by("state").agg(next_med=pl.col("price").median(), next_mean=pl.col("price").mean(),
                                  next_geo=pl.col("lp").mean().exp() - 1, n=pl.len()).sort("state"))
chain

state,next_med,next_mean,next_geo,n
enum,f64,f64,f64,u32
"""free""",9.990,12.556,4.991,159473
"""$0-5""",9.990,12.604,7.348,94037
"""$5-10""",9.990,14.633,9.171,178239
"""$10-20""",14.990,17.188,10.846,336240
"""$20-30""",19.990,19.681,12.522,103479
"""$30-60""",19.990,21.150,13.301,84977
"""$60+""",14.990,17.133,9.697,2892


The **bucket Markov chain** uses the price bucket of the user's last review day as its state. From each state
it predicts the training median, mean or geometric mean of the next price, whichever matches the metric (one
row for each). Higher states lead to higher next prices, but the spread between states is much smaller than
the spread between the buckets themselves.

In [10]:
def chain_pred(d, col):
    return d.with_columns(state=bucket_of("last_p")).join(chain, on="state", how="left")[col].to_numpy()

lin = LinearRegression().fit(tr.select(HIST).to_numpy(), np.log1p(y_tr))

def baselines(d, y_fit):
    return {
        "constant: median": np.full(d.height, np.median(y_fit)),
        "constant: mean": np.full(d.height, y_fit.mean()),
        "constant: geometric": np.full(d.height, np.expm1(np.log1p(y_fit).mean())),
        "last price": d["last_p"].to_numpy(),
        "user median (last 10)": d["med10"].to_numpy(),
        f"EWA (alpha={A_EWA})": d[f"ewm{A_EWA}"].to_numpy(),
        f"shrunk user mean (k={K_SHRINK})": shrunk(d, K_SHRINK),
        "Markov chain on buckets: median": chain_pred(d, "next_med"),
        "Markov chain on buckets: mean": chain_pred(d, "next_mean"),
        "Markov chain on buckets: geometric": chain_pred(d, "next_geo"),
        "Markov chain on games": np.expm1(d["tr_lp"].to_numpy()),
        "linear regression (history)": np.expm1(lin.predict(d.select(HIST).to_numpy())),
    }

base_val = table(baselines(va, y_tr), y_va)
base_val

model,MAE,MSE,MSLE
str,f64,f64,f64
"""constant: median""",12.673,319.589,1.809
"""constant: mean""",12.701,312.164,1.861
"""constant: geometric""",13.923,397.033,1.696
"""last price""",14.542,465.547,2.462
"""user median (last 10)""",12.952,381.995,2.037
"""EWA (alpha=0.1)""",12.948,375.114,1.955
"""shrunk user mean (k=5)""",13.124,368.758,1.520
"""Markov chain on buckets: median""",12.235,311.857,1.670
"""Markov chain on buckets: mean""",12.267,296.211,1.782


History-only baselines struggle. On validation the user's own last price is worse than a constant on every
metric, and our EWA only roughly ties the constants. Two models clearly beat the constants on MSLE: the
**game-level Markov chain** (users who just reviewed the same game tend to review games at similar prices
next) and the **linear regression** on history features.

---

## 5. Models

We combine all feature groups with two model families:

* **Ridge regression** on standardised features with a log target, the course's linear model. It is fast and
  interpretable, but it can only weight each feature linearly and needs NaNs filled.
* **Gradient-boosted trees** (`HistGradientBoostingRegressor`, as in hw3), one per metric: squared error on
  $\log(1+y)$ for MSLE, absolute error on $y$ for MAE, squared error on $y$ for MSE. Trees handle the
  nonlinear interactions (for example, a free-game history means more for a user with many reviews), handle
  missing values natively, and can use the matching loss directly. The cost is less interpretability and
  slower training.

### Feature ablation on a sample

Feature groups are added one at a time. Each model is trained on 250k sampled training targets and
scored on the full validation set.

In [11]:
HEADS = {
    "GBM log (MSLE)": dict(loss="squared_error", target="log"),
    "GBM abs (MAE)": dict(loss="absolute_error", target="raw"),
    "GBM sq (MSE)": dict(loss="squared_error", target="raw"),
}

def fit_head(head, A, y, **params):
    spec = HEADS[head]
    m = HistGradientBoostingRegressor(loss=spec["loss"], random_state=SEED, **params)
    m.fit(A, np.log1p(y) if spec["target"] == "log" else y)
    return m

def predict_head(head, m, B):
    p = m.predict(B)
    return np.expm1(p) if HEADS[head]["target"] == "log" else p

sample = tr.sample(250_000, seed=SEED)
y_s = sample["price"].to_numpy()
GROUPS = {"history": HIST, "+ taste": HIST + TASTE, "+ timing": HIST + TASTE + TIMING,
          "+ Markov": HIST + TASTE + TIMING + MARKOV, "+ global": FEATURES,
          "+ account size (crawl-time)": FEATURES + ["lprod"]}
rows = []
for gname, cols in GROUPS.items():
    A, B = sample.select(cols).to_numpy(), va.select(cols).to_numpy()
    for head in HEADS:
        m = fit_head(head, A, y_s, max_iter=300)
        rows.append({"features": gname, "head": head, **scores(y_va, predict_head(head, m, B))})
ablation = pl.DataFrame(rows)
def own_metric(d):
    pick = lambda metric, head: pl.col(metric).filter(pl.col("head") == head).first().alias(f"{metric} ({head})")
    return d.group_by("features", maintain_order=True).agg(
        pick("MSLE", "GBM log (MSLE)"), pick("MAE", "GBM abs (MAE)"), pick("MSE", "GBM sq (MSE)"))

own_metric(ablation)

features,MSLE (GBM log (MSLE)),MAE (GBM abs (MAE)),MSE (GBM sq (MSE))
str,f64,f64,f64
"""history""",1.502,11.866,284.743
"""+ taste""",1.488,11.846,284.153
"""+ timing""",1.481,11.768,281.838
"""+ Markov""",1.463,11.705,277.135
"""+ global""",1.457,11.737,273.728
"""+ account size (crawl-time)""",1.422,11.608,271.893


Reading each head on its own metric:

* **History alone** already beats every baseline. Most of the gain comes from letting a model combine
  statistics that are weak on their own.
* **Taste and timing** help MSLE and MAE a little.
* **Markov transitions and global context** help mostly the log head (MSLE), and they hurt the squared-error
  head on MSE slightly. Trees cannot extrapolate past the training range of the global price, which keeps
  rising.
* **Account size** (`products`, the number of games the account owns) is the strongest single addition. But
  it was recorded when the crawl ran, after most reviews were written, so for a 2015 review it partly
  describes the future. We report it as an ablation and leave it out of the main model.

### Latent-factor taste (ALS): a negative result

The course's implicit-feedback matrix factorisation could capture taste that genres miss. We fit ALS on
user × game interactions **before July 2016 only**. Each user is then represented by the mean factor vector
of their past games, and those 16 numbers are added as features. To keep this honest, the GBM is trained on
targets after the ALS cutoff.

In [12]:
CUT = dt.date(2016, 7, 1)
old = ev.filter(pl.col("day") < CUT)
items = old["product_id"].unique().sort()
users_old = old["user_id"].unique()
item_ix = dict(zip(items, range(len(items))))
user_ix = dict(zip(users_old, range(len(users_old))))
R = sp.csr_matrix((np.ones(old.height, dtype=np.float32),
                   ([user_ix[x] for x in old["user_id"]], [item_ix[x] for x in old["product_id"]])),
                  shape=(len(users_old), len(items)))
als = AlternatingLeastSquares(factors=16, regularization=0.1, alpha=10, iterations=15, random_state=SEED)
with threadpool_limits(1, "blas"):
    als.fit(R, show_progress=False)

E = [f"als{k}" for k in range(16)]
emb = pl.DataFrame({"product_id": items, "known": np.ones(len(items)),
                    **{c: als.item_factors[:, k] for k, c in enumerate(E)}})
taste = (ev.select("user_id", "product_id", "day").join(emb, on="product_id", how="left")
           .with_columns(pl.col(E + ["known"]).fill_null(0))
           .with_columns(pl.col(E + ["known"]).cum_sum().over(u))
           .group_by("user_id", "day", maintain_order=True).agg(pl.col(E + ["known"]).last())
           .with_columns(pl.col(E + ["known"]).shift(1).over(u))
           .with_columns(pl.col(E) / pl.col("known")))

late = tr.filter(pl.col("day") >= CUT).sample(250_000, seed=SEED).join(taste, on=["user_id", "day"], how="left")
va_als = va.join(taste, on=["user_id", "day"], how="left")
rows = []
for gname, cols in {"all features": FEATURES, "all + ALS taste": FEATURES + E}.items():
    for head in HEADS:
        m = fit_head(head, late.select(cols).to_numpy(), late["price"].to_numpy(), max_iter=300)
        rows.append({"features": gname, "head": head, **scores(y_va, predict_head(head, m, va_als.select(cols).to_numpy()))})
own_metric(pl.DataFrame(rows))

/Users/jedwin321/miniforge3/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 10 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


features,MSLE (GBM log (MSLE)),MAE (GBM abs (MAE)),MSE (GBM sq (MSE))
str,f64,f64,f64
"""all features""",1.459,11.718,273.256
"""all + ALS taste""",1.457,11.730,273.403


The ALS vectors leave every metric essentially unchanged. The taste they capture is already covered by the
genre shares, popularity and Markov features, so we drop ALS from the final model. (Training only on targets
after July 2016 gives nearly the same scores as training on all years, so drift between years is not what
limits the model.)

### Review text (TF-IDF): a second negative result

A user's earlier reviews might show their attitude to price ("overpriced", "wait for a sale", "f2p").
Only text from **earlier days** is used. The target review's own text describes the target game, so using it
would leak the answer.

* Text is lowercased, split into words and stemmed with the Porter stemmer. TF-IDF uses unigrams and bigrams,
  and the vocabulary is learned from reviews before July 2016.
* A ridge regression maps the average TF-IDF vector of a user's last 10 earlier reviews to the log price of
  their next game. It is fit on 60k targets before July 2016. The ridge is linear, so its prediction for an
  average vector equals the average of its per-review scores. That lets us build the feature with the same
  state-before-the-day pattern.
* We also add the running share of earlier reviews that mention price words.

The parquet cache dropped review text, so the text is re-read for a 25% sample of users (those whose
`user_id` is divisible by 4). The experiment then runs on that sample.

In [13]:
TEXT = DATA / "cache" / "review_text_q4.parquet"
if not TEXT.exists():
    uid_re = re.compile(r"u'user_id': u'(\d+)'")
    rows = []
    with open(DATA / "steam_reviews.json", encoding="utf-8", errors="replace") as f:
        for line in f:
            m = uid_re.search(line)
            if m and int(m.group(1)) % 4 == 0:
                r = ast.literal_eval(line)
                rows.append((r["user_id"], r["product_id"], r["date"], r.get("text") or ""))
    (pl.DataFrame(rows, schema=["user_id", "product_id", "day", "text"], orient="row")
       .with_columns(pl.col("day").str.to_date(strict=False)).unique().write_parquet(TEXT))

q4 = pl.col("user_id").cast(pl.UInt64) % 4 == 0
rt = (pl.read_parquet(TEXT).join(ev.filter(q4).select("user_id", "product_id"), on=["user_id", "product_id"], how="semi")
        .sort("user_id", "day", "product_id").unique(["user_id", "product_id"], keep="first", maintain_order=True)
        .with_row_index("row"))

stem = lru_cache(maxsize=None)(PorterStemmer().stem)
word_re = re.compile(r"[a-z0-9$']+")
docs = [[stem(w) for w in word_re.findall(s.lower())] for s in rt["text"].to_list()]
with_bigrams = lambda d: d + [a + "_" + b for a, b in zip(d, d[1:])]
vec = TfidfVectorizer(analyzer=with_bigrams, min_df=20, max_features=30_000, sublinear_tf=True)
vec.fit([docs[i] for i in rt.filter(pl.col("day") < CUT)["row"].to_list()])
W = vec.transform(docs).tocsr()
print(f"{rt.height:,} reviews from {rt['user_id'].n_unique():,} users, vocabulary {W.shape[1]:,}")

699,627 reviews from 369,184 users, vocabulary 30,000


In [14]:
fit_t = X.filter(q4 & (pl.col("day") < CUT)).sample(60_000, seed=SEED).select("user_id", "day", "lp").with_row_index("t")
window = (fit_t.join(rt.select("user_id", pl.col("day").alias("d"), "row"), on="user_id")
               .filter(pl.col("d") < pl.col("day")).sort("t", "d", "row")
               .group_by("t", maintain_order=True).tail(10)
               .with_columns(w=1 / pl.len().over("t")))
Avg = sp.csr_matrix((window["w"].to_numpy(), (window["t"].to_numpy(), window["row"].to_numpy())),
                    shape=(fit_t.height, W.shape[0])) @ W
text_ridge = Ridge(alpha=3.0).fit(Avg, fit_t["lp"].to_numpy())

names = vec.get_feature_names_out()
order = np.argsort(text_ridge.coef_)
print("words before cheaper games:  ", ", ".join(names[order[:15]]))
print("words before pricier games:  ", ", ".join(names[order[-15:]]))

words before cheaper games:   tf2, server, minecraft, soo, clicker, f2p, viru, uninstal, mmo, fun_great, free, and_i, anim, in_game, and_mani
words before pricier games:   civ, worth_it, town, unit, also_ha, 50, much_better, stop_play, it_simpl, war, a_ing, dark_soul, orc, xcom, server_to


In [15]:
PRICE_WORDS = ["price", "sale", "worth", "money", "cheap", "expens", "refund", "free", "f2p", "pay", "overpr", "discount", "$"]
mentions = {f"w_{w.strip('$') or 'dollar'}": np.array([any(t.startswith(w) for t in d) for d in docs], dtype=float)
            for w in PRICE_WORDS}
WORDS = list(mentions)
text_state = (rt.with_columns(score=pl.Series(W @ text_ridge.coef_ + text_ridge.intercept_),
                              **{k: pl.Series(v) for k, v in mentions.items()})
                .with_columns(txt=pl.col("score").rolling_mean(10, min_samples=1).over(u),
                              k=pl.int_range(1, pl.len() + 1).over(u),
                              **{w: pl.col(w).cum_sum().over(u) for w in WORDS})
                .group_by("user_id", "day", maintain_order=True).agg(pl.col(["txt", "k"] + WORDS).last())
                .with_columns(pl.col(["txt", "k"] + WORDS).shift(1).over(u))
                .with_columns(pl.col(WORDS) / pl.col("k")))

late_q = X.filter(q4 & (pl.col("split") == "train") & (pl.col("day") >= CUT)).join(text_state, on=["user_id", "day"], how="left")
va_q = X.filter(q4 & (pl.col("split") == "val")).join(text_state, on=["user_id", "day"], how="left")
rows = []
for gname, cols in {"all features": FEATURES, "+ TF-IDF score": FEATURES + ["txt"],
                    "+ price words": FEATURES + WORDS, "+ both": FEATURES + ["txt"] + WORDS}.items():
    for head in HEADS:
        m = fit_head(head, late_q.select(cols).to_numpy(), late_q["price"].to_numpy(), max_iter=300)
        rows.append({"features": gname, "head": head,
                     **scores(va_q["price"].to_numpy(), predict_head(head, m, va_q.select(cols).to_numpy()))})
alone = scores(va_q["price"].to_numpy(), np.expm1(va_q["txt"].fill_null(PRIOR["lp"]).to_numpy()))
print(f"TF-IDF score alone on validation: MSLE {alone['MSLE']:.3f}")
own_metric(pl.DataFrame(rows))

TF-IDF score alone on validation: MSLE 1.639


features,MSLE (GBM log (MSLE)),MAE (GBM abs (MAE)),MSE (GBM sq (MSE))
str,f64,f64,f64
"""all features""",1.456,11.678,285.134
"""+ TF-IDF score""",1.454,11.681,284.100
"""+ price words""",1.456,11.727,284.381
"""+ both""",1.453,11.724,284.422


On its own, the text score (MSLE 1.64) beats the raw history averages such as EWA (1.96), though not the
shrunk mean or the linear regression. Its strongest words are game names: *tf2*, *minecraft* and *f2p* come before cheap games, *dark souls*, *civ* and *xcom* before expensive
ones. The model already has those games' exact prices and genres, so adding the text barely moves any
metric. Text is left out of the final model.

### Tuning

A small grid per head, trained on the same 250k sample and scored on validation.

In [16]:
GRID = [dict(learning_rate=lr, max_leaf_nodes=leaves, min_samples_leaf=leaf, max_iter=400)
        for lr in (0.05, 0.1) for leaves in (31, 127) for leaf in (50, 500)]
A, B = sample.select(FEATURES).to_numpy(), va.select(FEATURES).to_numpy()
rows = []
for head in HEADS:
    metric = head.split("(")[1][:-1]
    for i, params in enumerate(GRID):
        m = fit_head(head, A, y_s, **params)
        rows.append({"head": head, "config": i, **params, metric: scores(y_va, predict_head(head, m, B))[metric]})
tuning = pl.DataFrame(rows)
BEST = {}
for head in HEADS:
    metric = head.split("(")[1][:-1]
    best = tuning.filter(pl.col("head") == head).sort(metric).row(0, named=True)
    BEST[head] = GRID[best["config"]]
    print(f"{head:15s} best {metric} {best[metric]:.3f} with {BEST[head]}")

GBM log (MSLE)  best MSLE 1.454 with {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 500, 'max_iter': 400}
GBM abs (MAE)   best MAE 11.698 with {'learning_rate': 0.05, 'max_leaf_nodes': 127, 'min_samples_leaf': 500, 'max_iter': 400}
GBM sq (MSE)    best MSE 272.722 with {'learning_rate': 0.1, 'max_leaf_nodes': 31, 'min_samples_leaf': 500, 'max_iter': 400}


### Fit on all of train, check on validation

In [17]:
A_tr, A_va = tr.select(FEATURES).to_numpy(), va.select(FEATURES).to_numpy()
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(np.nan_to_num(A_tr), np.log1p(y_tr))
models = {head: fit_head(head, A_tr, y_tr, **BEST[head]) for head in HEADS}

def main_preds(d, ridge, models):
    A = d.select(FEATURES).to_numpy()
    out = {"ridge (all features, log)": np.expm1(ridge.predict(np.nan_to_num(A)))}
    out.update({head: predict_head(head, m, A) for head, m in models.items()})
    return out

val_preds = {**baselines(va, y_tr), **main_preds(va, ridge, models)}
val_table = table(val_preds, y_va)
val_table

model,MAE,MSE,MSLE
str,f64,f64,f64
"""constant: median""",12.673,319.589,1.809
"""constant: mean""",12.701,312.164,1.861
"""constant: geometric""",13.923,397.033,1.696
"""last price""",14.542,465.547,2.462
"""user median (last 10)""",12.952,381.995,2.037
"""EWA (alpha=0.1)""",12.948,375.114,1.955
"""shrunk user mean (k=5)""",13.124,368.758,1.520
"""Markov chain on buckets: median""",12.235,311.857,1.670
"""Markov chain on buckets: mean""",12.267,296.211,1.782


### Ensemble

The models make different mistakes, so a weighted blend might beat the best single head. There is one blend
per metric, and each is fit with that metric's own loss:

* **MSLE:** non-negative least squares on $\log(1+\hat y)$ of each member;
* **MSE:** non-negative least squares on raw predictions;
* **MAE:** non-negative weights that minimise absolute error directly.

Every blend also gets an intercept. The members are the three GBM heads, the ridge, the game-level Markov
chain and the shrunk user mean.

The weights can't be fit on the rows they're scored on, so we split the validation users in two. We fit on
one half, score on the other, then swap halves and average the two scores.

In [18]:
MEMBERS = ["GBM log (MSLE)", "GBM abs (MAE)", "GBM sq (MSE)", "ridge (all features, log)",
           "Markov chain on games", f"shrunk user mean (k={K_SHRINK})"]
OWN_HEAD = {"MAE": "GBM abs (MAE)", "MSE": "GBM sq (MSE)", "MSLE": "GBM log (MSLE)"}

def design(metric, P):
    P = np.clip(P, 0, None)
    return np.column_stack([np.log1p(P) if metric == "MSLE" else P, np.ones(len(P))])

def blend_fit(metric, P, y):
    Z = design(metric, P)
    if metric == "MSLE":
        return nnls(Z, np.log1p(y))[0]
    if metric == "MSE":
        return nnls(Z, y)[0]
    w0 = np.r_[np.zeros(P.shape[1]), np.median(y)]
    w0[MEMBERS.index(OWN_HEAD["MAE"])] = 1.0
    bounds = [(0, None)] * P.shape[1] + [(None, None)]
    return minimize(lambda w: np.mean(np.abs(y - Z @ w)), w0, method="Powell", bounds=bounds).x

def blend_apply(metric, P, w):
    out = design(metric, P) @ w
    return np.expm1(out) if metric == "MSLE" else out

P_va = np.column_stack([val_preds[m] for m in MEMBERS])
half = (va["user_id"].cast(pl.UInt64) % 2 == 0).to_numpy()
rows = []
for metric in OWN_HEAD:
    single, blended = [], []
    for fit_on in (half, ~half):
        w = blend_fit(metric, P_va[fit_on], y_va[fit_on])
        score_on = ~fit_on
        single.append(scores(y_va[score_on], val_preds[OWN_HEAD[metric]][score_on])[metric])
        blended.append(scores(y_va[score_on], blend_apply(metric, P_va[score_on], w))[metric])
    rows.append({"metric": metric, "best single head": np.mean(single), "ensemble (held-out half)": np.mean(blended)})
WEIGHTS = {metric: blend_fit(metric, P_va, y_va) for metric in OWN_HEAD}
display(pl.DataFrame(rows))
pl.DataFrame({"member": MEMBERS + ["intercept"], **{m: w for m, w in WEIGHTS.items()}})

metric,best single head,ensemble (held-out half)
str,f64,f64
"""MAE""",11.685,11.676
"""MSE""",272.829,271.748
"""MSLE""",1.454,1.453


member,MAE,MSE,MSLE
str,f64,f64,f64
"""GBM log (MSLE)""",0.219,0.016,0.846
"""GBM abs (MAE)""",0.280,0.156,0.000
"""GBM sq (MSE)""",0.509,0.603,0.000
"""ridge (all features, log)""",0.062,0.172,0.107
"""Markov chain on games""",0.005,0.082,0.000
"""shrunk user mean (k=5)""",0.004,0.136,0.043
"""intercept""",-1.849,0.356,0.000


On the held-out half, blending improves MAE and MSE by well under 1% and leaves MSLE unchanged. The MSLE
blend puts almost all its weight on the log head. The heads already use the same features, so there is
little left for a blend to add. We still carry the ensembles to test, and the bootstrap checks whether they
beat the single heads there.

---

## 6. Test

Everything is refit on train + validation, using the hyperparameters chosen above. This includes the
constants, the Markov chain table, the linear models and the GBMs, but not the features: they are already
time-honest for every row and don't change. Then each model is scored once on the test set.

For reference, the teammate's preliminary EWA reported MAE 13.17, MSE 343.1 and MSLE 1.80. Their evaluation
set may differ from ours, so our own EWA row is the like-for-like comparison.

In [19]:
full = pl.concat([tr, va])
y_full = full["price"].to_numpy()
g = np.log1p(y_full).mean()
chain = (full.with_columns(state=bucket_of("last_p"))
             .group_by("state").agg(next_med=pl.col("price").median(), next_mean=pl.col("price").mean(),
                                    next_geo=pl.col("lp").mean().exp() - 1, n=pl.len()))
lin = LinearRegression().fit(full.select(HIST).to_numpy(), np.log1p(y_full))
A_full = full.select(FEATURES).to_numpy()
ridge_f = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(np.nan_to_num(A_full), np.log1p(y_full))
models_f = {head: fit_head(head, A_full, y_full, **BEST[head]) for head in HEADS}

test_preds = {**baselines(te, y_full), **main_preds(te, ridge_f, models_f)}
P_te = np.column_stack([test_preds[m] for m in MEMBERS])
test_preds.update({f"ensemble ({metric})": blend_apply(metric, P_te, WEIGHTS[metric]) for metric in OWN_HEAD})
test_table = table(test_preds, y_te)
peer = pl.DataFrame([{"model": "teammate's EWA (reported, own protocol)", "MAE": 13.17258, "MSE": 343.13568, "MSLE": 1.80063}])
pl.concat([test_table, peer])

model,MAE,MSE,MSLE
str,f64,f64,f64
"""constant: median""",12.160,300.371,1.544
"""constant: mean""",12.189,289.245,1.596
"""constant: geometric""",13.734,382.032,1.517
"""last price""",14.917,462.716,2.458
"""user median (last 10)""",13.343,377.780,2.061
"""EWA (alpha=0.1)""",13.279,368.237,1.942
"""shrunk user mean (k=5)""",13.179,358.551,1.408
"""Markov chain on buckets: median""",11.974,294.240,1.458
"""Markov chain on buckets: mean""",11.934,276.721,1.541


### Is the improvement real?

We use a bootstrap over **users**, not reviews, because one user's reviews are correlated. For each metric we
compare that metric's ensemble against our EWA, the best constant and the single GBM head, and report the 95%
interval of the improvement on test.

In [20]:
uid = te["user_id"].to_numpy()
_, user_code = np.unique(uid, return_inverse=True)
n_users = user_code.max() + 1

def per_review_loss(metric, p):
    p = np.clip(p, 0, None)
    return {"MAE": np.abs(y_te - p), "MSE": (y_te - p) ** 2, "MSLE": (np.log1p(y_te) - np.log1p(p)) ** 2}[metric]

rng = np.random.default_rng(SEED)
draws = rng.multinomial(n_users, np.full(n_users, 1 / n_users), size=500)
cnt = np.bincount(user_code, minlength=n_users)
rows = []
for metric, head, const in [("MAE", "ensemble (MAE)", "constant: median"), ("MSE", "ensemble (MSE)", "constant: mean"),
                            ("MSLE", "ensemble (MSLE)", "constant: geometric")]:
    for ref in (f"EWA (alpha={A_EWA})", const, OWN_HEAD[metric]):
        diff = per_review_loss(metric, test_preds[ref]) - per_review_loss(metric, test_preds[head])
        s = np.bincount(user_code, weights=diff, minlength=n_users)
        boot = (draws @ s) / (draws @ cnt)
        rows.append({"metric": metric, "model": head, "vs": ref, "improvement": diff.mean(),
                     "ci_low": np.quantile(boot, 0.025), "ci_high": np.quantile(boot, 0.975)})
pl.DataFrame(rows)

metric,model,vs,improvement,ci_low,ci_high
str,str,str,f64,f64,f64
"""MAE""","""ensemble (MAE)""","""EWA (alpha=0.1)""",1.724,1.674,1.775
"""MAE""","""ensemble (MAE)""","""constant: median""",0.605,0.569,0.642
"""MAE""","""ensemble (MAE)""","""GBM abs (MAE)""",-0.031,-0.039,-0.023
"""MSE""","""ensemble (MSE)""","""EWA (alpha=0.1)""",112.223,95.071,133.486
"""MSE""","""ensemble (MSE)""","""constant: mean""",33.231,31.987,34.486
"""MSE""","""ensemble (MSE)""","""GBM sq (MSE)""",0.927,0.743,1.131
"""MSLE""","""ensemble (MSLE)""","""EWA (alpha=0.1)""",0.611,0.596,0.625
"""MSLE""","""ensemble (MSLE)""","""constant: geometric""",0.186,0.179,0.193
"""MSLE""","""ensemble (MSLE)""","""GBM log (MSLE)""",-0.001,-0.001,-0.000


Positive numbers mean the ensemble has lower error. Against EWA and against the best constant, the 95%
intervals are far from zero on all three metrics. Against the single GBM heads, the ensembles barely
differ: about 1 point better on MSE, a tie on MSLE, and slightly *worse* on MAE. Our final picks are the
abs head for MAE, the MSE ensemble for MSE, and the log head for MSLE.

### Where the model helps, and where it can't

We break the test error down by history length and by whether the user had reviewed anything before the
test period. Then we look at how predictions compare with true prices.

In [21]:
seg = te.select("n", "day", "user_id", "price").with_columns(
    history=pl.col("n").cut([1, 4, 19], labels=["1", "2-4", "5-19", "20+"]).cast(pl.Enum(["1", "2-4", "5-19", "20+"])),
    seen_before_test=pl.col("user_id").is_in(full["user_id"].implode()),
    p_log=test_preds["ensemble (MSLE)"], p_ewa=test_preds[f"EWA (alpha={A_EWA})"],
    p_const=test_preds["constant: geometric"])
sle = lambda c: (pl.col("price").log1p() - pl.col(c).clip(0).log1p()) ** 2
by_hist = (seg.group_by("history").agg(targets=pl.len(), MSLE_model=sle("p_log").mean(), MSLE_EWA=sle("p_ewa").mean(),
                                       MSLE_constant=sle("p_const").mean()).sort("history"))
by_seen = seg.group_by("seen_before_test").agg(targets=pl.len(), MSLE_model=sle("p_log").mean(),
                                                MSLE_EWA=sle("p_ewa").mean(), MSLE_constant=sle("p_const").mean())
display(by_hist, by_seen)

history,targets,MSLE_model,MSLE_EWA,MSLE_constant
enum,u32,f64,f64,f64
"""1""",50275,1.501,2.789,1.640
"""2-4""",47793,1.363,1.743,1.552
"""5-19""",34474,1.159,1.263,1.392
"""20+""",10663,0.948,1.034,1.184


seen_before_test,targets,MSLE_model,MSLE_EWA,MSLE_constant
bool,u32,f64,f64,f64
false,60514,1.480,2.614,1.631
true,82691,1.223,1.450,1.434


The model's advantage over EWA is largest for users with a single past review (MSLE 1.50 vs 2.79). There,
EWA simply repeats one price, while the model pulls the prediction toward the population. With longer
histories every method improves, and the model stays ahead. Users who reviewed nothing before the test
period are harder for every method, but the model still beats the constant for them.

In [22]:
calib = (seg.with_columns(true_bucket=bucket_of("price"))
            .group_by("true_bucket").agg(targets=pl.len(), true_median=pl.col("price").median(),
                                         predicted_median=pl.col("p_log").median(),
                                         MSLE=sle("p_log").mean()).sort("true_bucket"))
calib

true_bucket,targets,true_median,predicted_median,MSLE
enum,u32,f64,f64,f64
"""free""",20309,0.000,8.866,5.191
"""$0-5""",10572,3.990,9.546,1.060
"""$5-10""",21489,9.990,10.997,0.174
"""$10-20""",49208,19.990,12.069,0.330
"""$20-30""",17765,29.990,13.393,0.714
"""$30-60""",23650,44.990,14.021,1.706
"""$60+""",212,79.990,11.843,5.083


In [23]:
long = calib.select("true_bucket", "true_median", "predicted_median").unpivot(index="true_bucket").with_columns(
    pl.col("true_bucket").cast(pl.Utf8))
long.hvplot.bar(x="true_bucket", y="value", by="variable", width=760, height=280, ylabel="price ($)",
                xlabel="true price bucket", title="Median true vs predicted price by true price bucket (test, MSLE ensemble)")

:Bars   [true_bucket,variable]   (value)

Predictions are squeezed into roughly \$9 to \$14. The model rarely knows a user's next game well enough to
predict \$0 or \$45. It moves in the right direction (higher true prices get higher predictions), but most of
the remaining error sits in the two tails. Free games are 14% of test targets, and their average squared log
error is over 5. That is the cost of not knowing which game comes next.

### What the model uses

Permutation importance for the log head on 30k validation targets: how much validation MSLE rises when one
feature is shuffled.

In [24]:
pi_rows = va.sample(30_000, seed=SEED)
def msle_scorer(est, A, y):
    return -np.mean((np.log1p(y) - np.log1p(np.clip(np.expm1(est.predict(A)), 0, None))) ** 2)
pi = permutation_importance(models["GBM log (MSLE)"], pi_rows.select(FEATURES).to_numpy(), pi_rows["price"].to_numpy(),
                            scoring=msle_scorer, n_repeats=3, random_state=SEED)
imp = pl.DataFrame({"feature": FEATURES, "MSLE_increase": pi.importances_mean}).sort("MSLE_increase", descending=True)
imp.head(15).hvplot.barh("feature", "MSLE_increase", width=600, height=380, flip_yaxis=True,
                         title="Permutation importance (log head, validation)")

:Bars   [feature]   (MSLE_increase)

---

## 7. Summary

| test set | MAE | MSE | MSLE |
|---|---|---|---|
| best constant | 12.16 | 289.2 | 1.517 |
| our EWA ($\alpha=0.1$) | 13.28 | 368.2 | 1.942 |
| teammate's EWA (reported) | 13.17 | 343.1 | 1.801 |
| **ours** | **11.52** (abs head) | **256.0** (MSE ensemble) | **1.331** (log head) |

* **History alone is weak.** On this split a constant beats the last price, the user's median and EWA on
  every metric. Averaging a user's past prices mostly adds noise. Shrinking toward the population helps
  (hw2's shrunk mean is the best simple baseline on MSLE).
* **The gains come from combining signals.** History, taste, timing, game-to-game transitions (the Markov
  chain) and recent global prices together cut test MSLE by 31% against our EWA and 12% against the best
  constant. Every interval excludes zero.
* **Match the model to the metric.** Each metric has a different best answer (median, mean, geometric mean),
  and a head trained for one metric loses on the others. That is why we report three models, not one.
* **Negative results.** ALS taste vectors, TF-IDF of past reviews and ensembling each change the results by
  less than 1%. Their information is already in the history and Markov features.
* **What's left is hard.** Predictions stay between about \$9 and \$14. Free games and \$40+ titles carry
  most of the error, because without knowing the next game the model can't commit to either tail.
* **Caveats.** The target is the game's list price when the data was crawled, not what the user paid. Account
  size would help (MSLE 1.457 → 1.423 on validation), but it was recorded at crawl time, so we leave it out.